# Phase 1: EDA

Answers the 5 Phase 1 questions from the project plan. Heavy lifting lives in `src/rogii_wellbore/`; this notebook drives, visualizes, and records findings.

**Questions:**
1. Eval-zone size per well — distribution of `TVT_input.isna()` count and span
2. GR comparability across wells — per-well GR distributions; per-well normalization needed?
3. Typewell TVT coverage — does typewell cover lateral well TVT range?
4. MD step uniformity — within-well and across-well
5. Train/test well overlap — already established (3 test wells = first 3 train wells)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rogii_wellbore import data
from rogii_wellbore.config import RAW_DIR

print("python :", sys.executable)
print("raw_dir:", RAW_DIR)

## Q1. Eval-zone size per well

**What we're checking:** For each of the 773 training wells, how big is the masked region we'd need to predict on (the `TVT_input.isna()` rows), and is the mask always a contiguous tail, or can it appear mid-well?

**Why it matters:** The shape of the mask determines what kind of problem this is. A random scatter of missing rows is interpolation/infilling. A contiguous tail is **forward extrapolation** — the model only ever predicts beyond what it can see, and autoregressive / causal architectures are natural choices. The size distribution also tells us how much context the model gets per well, and how far ahead it has to predict.

In [ ]:
hw_train = data.load_horizontal("train")
print(hw_train.shape)
hw_train.head(2)

In [ ]:
def well_eval_stats(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    mask = g["TVT_input"].isna()
    n_eval = int(mask.sum())
    if n_eval == 0:
        return pd.Series(
            {
                "n_rows": n,
                "n_eval": 0,
                "n_known": n,
                "eval_frac": 0.0,
                "eval_start": np.nan,
                "eval_end": np.nan,
                "eval_contiguous": True,
            }
        )
    eval_idx = np.where(mask.values)[0]
    start, end = int(eval_idx[0]), int(eval_idx[-1])
    contiguous = (end - start + 1) == n_eval
    return pd.Series(
        {
            "n_rows": n,
            "n_eval": n_eval,
            "n_known": n - n_eval,
            "eval_frac": n_eval / n,
            "eval_start": start,
            "eval_end": end,
            "eval_contiguous": contiguous,
        }
    )


eval_stats = hw_train.groupby("well_id").apply(well_eval_stats, include_groups=False)
eval_stats.head()

In [ ]:
print(f"wells: {len(eval_stats)}")
print(f"all contiguous tail eval zones: {eval_stats['eval_contiguous'].all()}")
print(f"any well with 0 eval rows: {(eval_stats['n_eval'] == 0).sum()}")
print(f"any well with 0 known rows: {(eval_stats['n_known'] == 0).sum()}")
print()
print("summary stats:")
eval_stats[["n_rows", "n_eval", "n_known", "eval_frac"]].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
)

In [ ]:
ends_at_last = eval_stats["eval_end"] == eval_stats["n_rows"] - 1
print(f"eval zone ends at last row: {ends_at_last.sum()} / {len(eval_stats)}")
if not ends_at_last.all():
    print("\nwells where eval zone does NOT end at last row:")
    display(eval_stats[~ends_at_last].head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(eval_stats["n_rows"], bins=50)
axes[0].set_xlabel("rows per well")
axes[0].set_ylabel("count")
axes[0].set_title("Well length")

axes[1].hist(eval_stats["n_eval"], bins=50)
axes[1].set_xlabel("eval rows per well")
axes[1].set_title("Eval zone size (rows)")

axes[2].hist(eval_stats["eval_frac"], bins=50)
axes[2].set_xlabel("eval rows / total rows")
axes[2].set_title("Eval zone fraction")

plt.tight_layout()
plt.savefig("../reports/figures/01_eval_zone_distributions.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
eval_stats.to_parquet("../data/interim/eval_zone_stats.parquet")
print("saved:", "data/interim/eval_zone_stats.parquet", eval_stats.shape)

In [ ]:
Path("docs").mkdir(exist_ok=True)
Path("../docs/01_eda_notes.md").write_text("""# Phase 1: EDA notes

## Q1. Eval-zone size per well

**All 773 train wells have the same eval-zone structure: a contiguous block of `TVT_input.isna()` rows at the tail of the well.** The mask never appears mid-well; it always extends from some row `k` to the final row.

This is forward extrapolation along the lateral, not random infilling. Implication for Phase 2: causal/autoregressive models are natural; no future leak during training.

### Distribution (773 wells)

| stat              | total rows | known rows | eval rows | eval fraction |
|-------------------|-----------:|-----------:|----------:|--------------:|
| min               |      2,058 |        851 |       407 |         0.198 |
| 5th pct           |      4,652 |      1,346 |     2,947 |         0.625 |
| median            |      6,576 |      1,703 |     4,840 |         0.740 |
| mean              |      6,588 |      1,692 |     4,895 |         0.733 |
| 95th pct          |      8,614 |      2,053 |     6,918 |         0.819 |
| max               |     12,141 |      2,392 |    10,052 |         0.875 |

### Observations
- **Known segment is tight** (851-2392 rows, std 217). Each well gives the model roughly the same amount of context.
- **Eval segment is wide** (407-10052 rows, std 1301). 25x range. A model needs to handle both short and very long forwad predictions.
- **One low-mask outlier** (eval_frac ≈ 0.20) — flagged but not investigated.
- Submission `id` integer = horizontal_well CSV row index; eval rows are exactly the rows where `TVT_input` is NaN.

### Figure
`reports/figures/01_eval_zone_distributions.png`
""")
print("wrote docs/01_eda_notes.md")

## Q2. GR comparability across wells

**What we're checking:** Are gamma-ray (GR) values comparable across the 773 wells, or do per-well means and scales differ enough that a raw `GR = 80` means different things in different wells? Also: how much GR data is actually present (NaN rate, gap structure)?

**Why it matters:** Most of the predictive signal lives in GR — it's the only continuous lithology proxy available at inference time. If wells are not directly comparable, any model trained on raw GR will overfit to well-specific GR baselines instead of learning a general formation-vs-GR relationship. The fix is per-well normalization, but it has a cost: it erases any cross-well signal in absolute GR. We need to quantify how much normalization is actually warranted.

In [ ]:
gr_stats = hw_train.groupby("well_id")["GR"].agg(
    [
        "count",
        "mean",
        "std",
        "min",
        "max",
        lambda s: s.quantile(0.05),
        lambda s: s.quantile(0.50),
        lambda s: s.quantile(0.95),
    ]
)
gr_stats.columns = ["n", "mean", "std", "min", "max", "p05", "p50", "p95"]
print("NaN GR rows:", hw_train["GR"].isna().sum())
gr_stats.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(gr_stats["mean"], bins=50)
axes[0].set_xlabel("per-well mean GR")
axes[0].set_ylabel("count")
axes[0].set_title("Distribution of per-well GR means")

axes[1].hist(gr_stats["std"], bins=50)
axes[1].set_xlabel("per-well std GR")
axes[1].set_title("Distribution of per-well GR stds")

# Per-well range (p95 - p05) gives a robust within-well spread
axes[2].scatter(gr_stats["mean"], gr_stats["p95"] - gr_stats["p05"], s=4, alpha=0.4)
axes[2].set_xlabel("per-well mean GR")
axes[2].set_ylabel("per-well p95 - p05 GR")
axes[2].set_title("Mean vs spread (per well)")

plt.tight_layout()
plt.savefig("../reports/figures/02_gr_per_well_stats.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
across_well_mean_std = gr_stats["mean"].std()  # how much per-well means vary
within_well_std_med = gr_stats["std"].median()  # typical per-well variation
ratio = across_well_mean_std / within_well_std_med

print(f"std of per-well means:    {across_well_mean_std:.2f}")
print(f"median of per-well stds:  {within_well_std_med:.2f}")
print(f"ratio (across / within):  {ratio:.2f}")
print()
print("Interpretation:")
print("  ratio << 1 -> wells are comparable as-is; per-well norm probably unnecessary")
print("  ratio ~ 1  -> per-well norm is sensible")
print("  ratio >> 1 -> per-well norm essential; or features become well-id-dependent")

In [ ]:
hw_train["gr_nan"] = hw_train["GR"].isna()
hw_train["in_eval"] = hw_train["TVT_input"].isna()

total = len(hw_train)
print(f"total rows:               {total:,}")
print(f"GR NaN:                   {hw_train['gr_nan'].sum():,} ({hw_train['gr_nan'].mean():.1%})")
print(f"GR NaN AND in eval zone:  {(hw_train['gr_nan'] & hw_train['in_eval']).sum():,}")
print(f"GR NaN AND in known zone: {(hw_train['gr_nan'] & ~hw_train['in_eval']).sum():,}")
print()
# Conditional rates
eval_rows = hw_train["in_eval"].sum()
known_rows = total - eval_rows
print(
    f"P(GR NaN | in eval zone):  {(hw_train['gr_nan'] & hw_train['in_eval']).sum() / eval_rows:.1%}"
)
print(
    f"P(GR NaN | in known zone): {(hw_train['gr_nan'] & ~hw_train['in_eval']).sum() / known_rows:.1%}"
)

# how many wells have ANY GR available in the eval zone?
gr_in_eval = hw_train[hw_train["in_eval"]].groupby("well_id")["GR"].apply(lambda s: s.notna().sum())
print(f"\nwells with >=1 GR value in eval zone: {(gr_in_eval > 0).sum()} / {len(gr_in_eval)}")
print(f"wells with ZERO GR in eval zone:      {(gr_in_eval == 0).sum()}")
gr_in_eval.describe()

In [ ]:
g = hw_train[hw_train["well_id"] == "000d7d20"].copy().reset_index(drop=True)
nan_mask = g["GR"].isna().values
# Run-length encoding of NaN runs
changes = np.diff(np.concatenate(([0], nan_mask.astype(int), [0])))
starts = np.where(changes == 1)[0]
ends = np.where(changes == -1)[0]
run_lengths = ends - starts
print(f"well 000d7d20: {nan_mask.sum()} / {len(nan_mask)} GR NaN ({nan_mask.mean():.1%})")
print(f"number of NaN runs: {len(run_lengths)}")
print("run-length distribution:")
print(pd.Series(run_lengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"\nfirst 20 run lengths: {run_lengths[:20].tolist()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(g["row_idx"], g["GR"], lw=0.4)
ax.axvline(
    g[g["TVT_input"].isna()]["row_idx"].iloc[0], color="red", ls="--", lw=1, label="eval start"
)
ax.set_xlabel("row_idx")
ax.set_ylabel("GR")
ax.set_title("GR vs row_idx, well 000d7d20")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/02b_gr_one_well.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
gr_nan_rate = hw_train.groupby("well_id")["GR"].apply(lambda s: s.isna().mean())
print("Per-well GR NaN rate distribution:")
print(gr_nan_rate.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
print(f"\nwells with > 50% GR NaN: {(gr_nan_rate > 0.5).sum()}")
print(f"wells with > 70% GR NaN: {(gr_nan_rate > 0.7).sum()}")

In [ ]:
worst = gr_nan_rate.sort_values(ascending=False).head(10)
print("10 worst wells by GR NaN rate:")
print(worst)

# save per-well GR stats for later phases
gr_stats["nan_rate"] = gr_nan_rate
gr_stats.to_parquet("../data/interim/gr_per_well_stats.parquet")
print(f"\nsaved: data/interim/gr_per_well_stats.parquet  {gr_stats.shape}")

In [ ]:
q2 = """

## Q2. GR comparability across wells

**Per-well normalization is sensible (and probably necessary).** The ratio of across-well variation to within-well variation is ≈ 0.89 — wells differ in mean GR about as much as a single well varies internally. A raw `GR = 80` means different things in different wells.

### Per-well summary (773 wells)

| stat            | mean GR | std GR | nan rate |
|-----------------|--------:|-------:|---------:|
| min             |    37.2 |    8.3 |    0.7%  |
| 5th pct         |    65.4 |   12.3 |    3.8%  |
| median          |    86.6 |   17.3 |   27.7%  |
| 95th pct        |   118.2 |   25.3 |   60.3%  |
| max             |   130.5 |   35.1 |   80.1%  |

- Across-well std of per-well means: 15.4
- Median of per-well stds: 17.3
- Ratio (across / within): **0.89** → per-well z-score is the right default

### Bimodality
The distribution of per-well GR means is bimodal (peaks ~85 and ~115), suggesting two clusters of wells (likely different formations, fields, or tool calibrations). A naive per-well z-score erases this cluster signal. Worth testing typewell-conditioned normalization in Phase 2 as an alternative.

### Missing GR
- **29.6% of GR values are NaN overall.** Rate is similar in known (23.5%) and eval (31.7%) zones — GR is *not* cut off at the eval boundary; it's intermittent tool dropouts.
- Per-well NaN rate varies widely: 1% (best) to 80% (worst), median 28%, std 19%.
- **145 wells (19%) have > 50% NaN**, 4 wells > 70%. Phase 2 should evaluate whether these are usable, downweight them, or drop them.
- NaN runs are short (median 1, 99th pct 11, max 19 rows on the spot-check well). Linear interpolation per well is safe; cap on max run-length to be confirmed across all wells.

### Outliers
Spot-checked well `000d7d20` shows a single-row GR spike to ~210 around row 4400 (vs ~95 baseline). Likely sensor glitch. Phase 2 needs robust scaling or outlier clipping.

### Figures
- `reports/figures/02_gr_per_well_stats.png` — distributions of per-well mean/std/spread
- `reports/figures/02b_gr_one_well.png` — GR trace for well 000d7d20 (gaps + outlier visible)
"""
Path("../docs/01_eda_notes.md").write_text(Path("../docs/01_eda_notes.md").read_text() + q2)
print("appended Q2 to docs/01_eda_notes.md")

## Q3. Typewell TVT coverage

**What we're checking:** Each lateral well comes paired with a vertical "typewell" reference (`TVT, GR, Geology`). Does each typewell's TVT range cover the corresponding lateral's TVT range, or are there laterals that wander into depths the typewell doesn't reach?

**Why it matters:** The typewell is the bridge between GR signal and depth. The strongest known feature for this problem (GR-template correlation against the typewell) matches each lateral row to the best-aligned typewell TVT. If the lateral wanders outside the typewell's TVT range, that feature has no candidate to match and must fall back. Knowing how often this happens, and by how much, tells us how robust the GR-match feature needs to be.

In [ ]:
tw_train = data.load_typewell("train")
print("typewell shape:", tw_train.shape)
print("NaN TVT in typewell:", tw_train["TVT"].isna().sum())
print("NaN GR in typewell: ", tw_train["GR"].isna().sum())

# Per-well TVT ranges from lateral (using TVT, since that's the truth we want covered)
lat_range = hw_train.groupby("well_id")["TVT"].agg(["min", "max", "count"]).add_prefix("lat_")
tw_range = tw_train.groupby("well_id")["TVT"].agg(["min", "max", "count"]).add_prefix("tw_")
cov = lat_range.join(tw_range, how="inner")
cov["lat_span"] = cov["lat_max"] - cov["lat_min"]
cov["tw_span"] = cov["tw_max"] - cov["tw_min"]
cov["below_gap"] = (cov["lat_min"] - cov["tw_min"]).clip(upper=0).abs()  # tw_min above lat_min
cov["above_gap"] = (cov["tw_max"] - cov["lat_max"]).clip(upper=0).abs()  # tw_max below lat_max
cov["covers_full"] = (cov["tw_min"] <= cov["lat_min"]) & (cov["tw_max"] >= cov["lat_max"])
cov.head()

In [ ]:
n = len(cov)
print(f"wells with typewell fully covering lateral TVT range: {cov['covers_full'].sum()} / {n}")
print(f"wells where typewell misses BELOW lat_min:            {(cov['below_gap'] > 0).sum()}")
print(f"wells where typewell misses ABOVE lat_max:            {(cov['above_gap'] > 0).sum()}")
print()
print("If misses, by how much (rows where gap > 0):")
print("  below_gap (ft):")
print(cov.loc[cov["below_gap"] > 0, "below_gap"].describe(percentiles=[0.5, 0.9, 0.99]))
print("  above_gap (ft):")
print(cov.loc[cov["above_gap"] > 0, "above_gap"].describe(percentiles=[0.5, 0.9, 0.99]))
print()
print("typewell length (n rows) distribution:")
print(cov["tw_count"].describe(percentiles=[0.05, 0.5, 0.95]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Coverage diagram: each well as a horizontal bar showing tw range, with lat range overlaid
sample = cov.sort_values("lat_min").iloc[::15]  # every 15th well, sorted by depth
y = np.arange(len(sample))
axes[0].hlines(y, sample["tw_min"], sample["tw_max"], color="C0", lw=2, label="typewell TVT")
axes[0].hlines(y + 0.3, sample["lat_min"], sample["lat_max"], color="C3", lw=2, label="lateral TVT")
axes[0].set_xlabel("TVT")
axes[0].set_ylabel("well (sample, sorted)")
axes[0].set_title("Typewell vs lateral TVT range (sampled)")
axes[0].legend()

axes[1].scatter(cov["lat_span"], cov["tw_span"], s=4, alpha=0.4)
m = max(cov["lat_span"].max(), cov["tw_span"].max())
axes[1].plot([0, m], [0, m], "k--", lw=0.5, label="y = x")
axes[1].set_xlabel("lateral TVT span")
axes[1].set_ylabel("typewell TVT span")
axes[1].set_title("Span comparison (above line = typewell wider)")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/03_typewell_coverage.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
problem_wells = cov[cov["below_gap"] > 0].sort_values("below_gap", ascending=False)
print(f"{len(problem_wells)} wells with typewell missing BELOW lat_min:")
print(problem_wells[["lat_min", "lat_max", "tw_min", "tw_max", "below_gap"]])

cov.to_parquet("../data/interim/typewell_coverage.parquet")
print(f"\nsaved: data/interim/typewell_coverage.parquet  {cov.shape}")

In [ ]:
q3 = """

## Q3. Typewell TVT coverage

**Typewell covers the lateral TVT range for 760 / 773 wells (98.3%).** Coverage is sufficient by default — the GR-template matching feature can find a valid TVT match for almost all eval rows.

### Coverage breakdown

| condition                                  | wells |
|--------------------------------------------|------:|
| typewell fully covers lateral TVT range    |   760 |
| typewell misses BELOW lat_min (gap > 0)    |    11 |
| typewell misses ABOVE lat_max (gap > 0)    |     2 |

- **Above gaps are negligible**: 2 wells with max 1.3 ft (less than typewell's 0.5 ft step).
- **Below gaps are real**: 11 wells with 117-645 ft of lateral depth not covered by the typewell (median 434 ft). For these wells, GR-template matching has no candidate match for the shallow-end rows. Falls back to anchor-based prediction is the right strategy.

### Typewell size distribution
- Median length 1874 rows (typewell step is 0.5 ft, so ~937 ft span)
- 5th-95th percentile: 1053-3918 rows
- Range: 636-10043 rows

### Span ratio
Across all wells, typewell span exceeds lateral span by a healthy margin (most points above y=x in the span plot). Cluster of wells at typewell_span ≈ 500 ft regardless of lateral span — looks like some typewells were clipped to a fixed window. Not a blocker.

### Action items for Phase 2
- The GR-template match feature must gracefully fall back when no typewell candidate exists in the search radius (the prior `predict_gr_match_v2` already does this — sets `pred = anchor`).
- Track the 11 below-coverage wells; their eval rows in the uncovered shallow zone will rely entirely on anchor/trajectory features, not typewell matching.

### Figure
`reports/figures/03_typewell_coverage.png` — coverage diagram + span comparison
"""
Path("../docs/01_eda_notes.md").write_text(Path("../docs/01_eda_notes.md").read_text() + q3)
print("appended Q3 to docs/01_eda_notes.md")

## Q4. MD step uniformity

**What we're checking:** Is the measured-depth (MD) step constant within each well? Is it the same across all 773 wells? Are MD values monotonically increasing without duplicates?

**Why it matters:** This is gotcha #6 from the project plan. Every windowed feature (rolling means, sliding-window correlations) and every sequence model (RNNs, transformers, 1D CNNs) implicitly assumes a uniform sampling grid. If MD spacing varies within or across wells, those operations either need an explicit resampling step, or features must carry the spacing along as an extra input. The answer here directly determines Phase 2 architecture options.

In [ ]:
# Are typewells shared across wells?
tw_signatures = tw_train.groupby("well_id")["TVT"].agg(["min", "max", "count"])
dup = tw_signatures.groupby(["min", "max", "count"]).size().sort_values(ascending=False)
print("Typewell signatures shared across N wells:")
print(dup.head(10))
print(f"\nunique typewell signatures: {len(dup)} / {len(tw_signatures)} wells")

In [ ]:
def md_step_stats(g: pd.DataFrame) -> pd.Series:
    md = g["MD"].values
    if len(md) < 2:
        return pd.Series(
            {
                "n": len(md),
                "min_step": np.nan,
                "max_step": np.nan,
                "median_step": np.nan,
                "any_neg": False,
                "any_dup": False,
                "n_unique_steps": 0,
            }
        )
    d = np.diff(md)
    return pd.Series(
        {
            "n": len(md),
            "min_step": float(d.min()),
            "max_step": float(d.max()),
            "median_step": float(np.median(d)),
            "any_neg": bool((d < 0).any()),  # non-monotonic
            "any_dup": bool((d == 0).any()),  # duplicate MDs
            "n_unique_steps": int(len(np.unique(np.round(d, 6)))),
        }
    )


md_stats = hw_train.groupby("well_id").apply(md_step_stats, include_groups=False)
print("Within-well MD step summary:")
print(md_stats.describe(percentiles=[0.05, 0.5, 0.95]))
print(f"\nwells with any negative step (non-monotonic): {md_stats['any_neg'].sum()}")
print(f"wells with any zero step (duplicate MD):      {md_stats['any_dup'].sum()}")
print(f"wells with constant step (n_unique_steps=1):  {(md_stats['n_unique_steps'] == 1).sum()}")

In [ ]:
print("Distribution of per-well median MD step:")
print(md_stats["median_step"].describe(percentiles=[0.05, 0.5, 0.95]))
print()
print("Value counts of per-well median MD step (rounded to 3dp):")
print(md_stats["median_step"].round(3).value_counts().head(10))

In [ ]:
nonuniform = md_stats[md_stats["n_unique_steps"] > 1].sort_values("max_step", ascending=False)
print(f"wells with non-constant MD step: {len(nonuniform)} / {len(md_stats)}")
if len(nonuniform) > 0:
    print("\nTop 10 by max_step:")
    print(nonuniform.head(10))
    print("\nDistribution of max_step (these wells):")
    print(nonuniform["max_step"].describe(percentiles=[0.5, 0.9, 0.99]))

In [ ]:
q4 = """

## Q4. MD step uniformity

**Perfect uniformity across all 773 wells.**

- MD step is exactly **1.0** in every well, every row.
- 0 wells with non-monotonic MD (no negative steps).
- 0 wells with duplicate MD (no zero steps).
- 773 / 773 wells have constant within-well step.
- 773 / 773 wells have the same step value (1.0).

### Implications for Phase 2
- **Windowed features, convolutions, RNNs/transformers all work on the uniform grid directly.** No resampling needed.
- `row_idx` is equivalent to `MD - MD_start` (a constant offset per well).
- The typewell is on a **0.5 ft** grid; cross-correlation with the lateral requires upsampling lateral (2x interpolation) or downsampling typewell (`np.interp` to a 1.0 ft grid). The prior `predict_gr_match_v2` uses the latter.
- Gotcha #6 from the project plan is resolved cleanly — no contingency code needed.
"""
Path("../docs/01_eda_notes.md").write_text(Path("../docs/01_eda_notes.md").read_text() + q4)
print("appended Q4 to docs/01_eda_notes.md")

## Q5. Train/test well overlap + geographic structure

**What we're checking:** Which wells appear in both train and test sets? What does the test schema look like? And spatially, where are these 773 wells — clustered in one field, or scattered across regions?

**Why it matters:** Two reasons. (a) The schema difference between train and test constrains which columns can be used as direct inputs at inference time (anything train-only becomes auxiliary supervision or feature engineering, not a model input). (b) Geographic structure affects CV design: if wells cluster in pads or fields, GroupKFold by `well_id` may overstate generalization, because nearby wells share geology / tool calibration. A pad-level grouped split would give a more pessimistic, leakage-safe estimate.

In [ ]:
train_wells = set(data.list_wells("train"))
test_wells = set(data.list_wells("test"))

overlap = train_wells & test_wells
print(f"train wells: {len(train_wells)}")
print(f"test wells:  {len(test_wells)}")
print(f"overlap:     {len(overlap)}  ({sorted(overlap)})")
print(f"test-only:   {len(test_wells - train_wells)}")
print(f"train-only:  {len(train_wells - test_wells)}")

In [ ]:
well_locs = (
    hw_train.groupby("well_id")
    .agg(
        x_start=("X", "first"),
        y_start=("Y", "first"),
        x_end=("X", "last"),
        y_end=("Y", "last"),
    )
    .reset_index()
)
well_locs["dx"] = well_locs["x_end"] - well_locs["x_start"]
well_locs["dy"] = well_locs["y_end"] - well_locs["y_start"]

print(f"X range: {well_locs['x_start'].min():.0f} to {well_locs['x_start'].max():.0f}")
print(f"Y range: {well_locs['y_start'].min():.0f} to {well_locs['y_start'].max():.0f}")
print(f"X span:  {well_locs['x_start'].max() - well_locs['x_start'].min():.0f}")
print(f"Y span:  {well_locs['y_start'].max() - well_locs['y_start'].min():.0f}")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(well_locs["x_start"], well_locs["y_start"], s=8, alpha=0.4, label="train wells")

# overlay the 3 test wells
test_locs = well_locs[well_locs["well_id"].isin(test_wells)]
ax.scatter(
    test_locs["x_start"],
    test_locs["y_start"],
    s=80,
    color="red",
    marker="x",
    label="test wells",
    zorder=5,
)

ax.set_xlabel("X (start)")
ax.set_ylabel("Y (start)")
ax.set_title("Well start locations (train + test overlap)")
ax.set_aspect("equal", adjustable="datalim")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/05_well_locations.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# crude region split at median X
well_locs["region"] = np.where(well_locs["x_start"] < well_locs["x_start"].median(), "SW", "NE")
well_locs = well_locs.merge(
    gr_stats[["mean"]].rename(columns={"mean": "gr_mean"}), left_on="well_id", right_index=True
)

fig, ax = plt.subplots(figsize=(8, 4))
for region, grp in well_locs.groupby("region"):
    ax.hist(grp["gr_mean"], bins=40, alpha=0.5, label=f"{region} (n={len(grp)})")
ax.set_xlabel("per-well mean GR")
ax.set_ylabel("count")
ax.set_title("GR mean by region (crude median-X split)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/05b_gr_by_region.png", dpi=100, bbox_inches="tight")
plt.show()

print("Per-region GR mean stats:")
print(well_locs.groupby("region")["gr_mean"].describe())

In [ ]:
well_locs.to_parquet("../data/interim/well_locations.parquet")
print(f"saved: data/interim/well_locations.parquet  {well_locs.shape}")

q5 = """

## Q5. Train/test well overlap + geographic structure

### Overlap
- **Train wells:** 773. **Test wells:** 3. **Overlap:** 3.
- The 3 test wells (`000d7d20`, `00bbac68`, `00e12e8b`) are exactly subsets of the train wells.
- **Schema differs:** test wells expose only `MD, X, Y, Z, GR, TVT_input`. The 6 surface-boundary columns (`ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA`) and `TVT` are train-only. These cannot be used as direct input features at inference time — only as auxiliary training targets.
- The same eval-zone tail mask appears in test as in train (3836 NaN rows in `000d7d20` test, identical to train). Public-LB evaluation is the masked tail of these 3 wells; ground truth held server-side.

### Geographic structure
- All 773 wells lie within a single field complex, ~33mi by 25mi (X: 2.86M to 3.04M, Y: 1.01M to 1.14M; units = ft).
- Wells cluster in **linear streaks** — these are pads/leases (multiple laterals drilled from a common surface location, branching out radially).
- Two visually distinct main concentrations: a SW cluster around (X≈2.88M, Y≈1.02M) and a NE cluster around (X≈3.00M, Y≈1.10M).
- All 3 test wells sit in the densest part of the NE cluster.

### Bimodality follow-up (from Q2)
A crude median-X split does NOT cleanly explain the bimodality of per-well mean GR (both halves have similar means ≈ 88 and contain the secondary peak ~115). The cluster signal appears to be finer-grained — likely **per-pad** rather than regional. A pad-level (X/Y proximity) cluster ID is a candidate engineered feature for Phase 2.

### Action items for Phase 2
- **CV strategy:** GroupKFold by `well_id` is the floor. Consider also a pad-level grouped split (cluster wells by spatial proximity) for a more pessimistic, leakage-safe estimate of generalization to new pads/fields.
- **Coordinate features:** raw X/Y as inputs is sensible; consider adding pad-cluster ID as a categorical.
- **Public-LB caveat:** the 3 test wells are interior to the densest training region. Private LB may include wells from sparser regions; generalization should be checked via pad-grouped CV before trusting public-LB scores.

### Figures
- `reports/figures/05_well_locations.png` — well map with test wells overlaid
- `reports/figures/05b_gr_by_region.png` — GR-by-region check (null result)
"""
Path("../docs/01_eda_notes.md").write_text(Path("../docs/01_eda_notes.md").read_text() + q5)
print("appended Q5 to docs/01_eda_notes.md")

## MLflow snapshot

Log Phase 1 summary stats and artifacts to MLflow as a dated audit trail. This gives later phases a reference point: "what did the data look like when EDA was done?"

In [ ]:
import mlflow

from rogii_wellbore.config import MLFLOW_TRACKING_URI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("eda")

with mlflow.start_run(run_name="phase1_data_snapshot"):
    # ... rest of the cell stays the same
    # Dataset-level
    mlflow.log_param("n_train_wells", 773)
    mlflow.log_param("n_test_wells", 3)
    mlflow.log_param("train_test_well_overlap", 3)
    mlflow.log_param("md_step", 1.0)
    mlflow.log_param("typewell_md_step", 0.5)
    mlflow.log_param("eval_zone_structure", "contiguous_tail")

    # Q1: eval-zone
    mlflow.log_metric("eval_zone_frac_median", 0.740)
    mlflow.log_metric("eval_zone_frac_min", 0.198)
    mlflow.log_metric("eval_zone_frac_max", 0.875)
    mlflow.log_metric("known_rows_median", 1703)
    mlflow.log_metric("known_rows_min", 851)

    # Q2: GR
    mlflow.log_metric("gr_across_within_ratio", 0.89)
    mlflow.log_metric("gr_nan_rate_overall", 0.296)
    mlflow.log_metric("gr_nan_rate_known", 0.235)
    mlflow.log_metric("gr_nan_rate_eval", 0.317)
    mlflow.log_metric("gr_wells_over_50pct_nan", 145)
    mlflow.log_metric("gr_wells_over_70pct_nan", 4)

    # Q3: typewell
    mlflow.log_metric("typewell_full_coverage_wells", 760)
    mlflow.log_metric("typewell_below_gap_wells", 11)
    mlflow.log_metric("typewell_above_gap_wells", 2)
    mlflow.log_metric("typewell_below_gap_median_ft", 434.23)

    # Q5: geographic
    mlflow.log_metric("unique_typewell_signatures", 752)
    mlflow.log_metric("field_x_span_ft", 177693)
    mlflow.log_metric("field_y_span_ft", 129771)

    # Artifacts: figures + interim tables
    for fig in [
        "../reports/figures/01_eval_zone_distributions.png",
        "../reports/figures/02_gr_per_well_stats.png",
        "../reports/figures/02b_gr_one_well.png",
        "../reports/figures/03_typewell_coverage.png",
        "../reports/figures/05_well_locations.png",
        "../reports/figures/05b_gr_by_region.png",
    ]:
        mlflow.log_artifact(fig, artifact_path="figures")

    for tbl in [
        "../data/interim/eval_zone_stats.parquet",
        "../data/interim/gr_per_well_stats.parquet",
        "../data/interim/typewell_coverage.parquet",
        "../data/interim/well_locations.parquet",
    ]:
        mlflow.log_artifact(tbl, artifact_path="tables")

    mlflow.log_artifact("../docs/01_eda_notes.md", artifact_path="docs")

print("logged Phase 1 EDA snapshot to MLflow.")